# Day 23 Solutions: Streaming Data Pipelines with Generators

## Lesson walkthrough

Streaming CSV cleaning; compose filter/map/batch generators.

The walkthrough cells run the lesson's original examples. The exercise answers after them contain the same code and reasoning as `day23_solutions.md`.


In [ ]:
from __future__ import annotations
from typing import Iterable, Iterator, Callable, TypeVar
import csv
from pathlib import Path
from itertools import islice

T = TypeVar("T")


def stream_csv(path: Path) -> Iterator[dict[str, str]]:
    with path.open(encoding="utf-8", newline="") as f:
        reader = csv.DictReader(f)
        for row in reader:
            yield row


def clean_rows(rows: Iterable[dict[str, str]]) -> Iterator[dict[str, str]]:
    for r in rows:
        r = {k: (v.strip() if isinstance(v, str) else v) for k, v in r.items()}
        if "code" in r and isinstance(r["code"], str):
            r["code"] = r["code"].upper()
        yield r


def only_if(rows: Iterable[T], pred: Callable[[T], bool]) -> Iterator[T]:
    for r in rows:
        if pred(r):
            yield r


def mapper(rows: Iterable[T], fn: Callable[[T], T]) -> Iterator[T]:
    for r in rows:
        yield fn(r)


def batched(rows: Iterable[T], size: int) -> Iterator[list[T]]:
    it = iter(rows)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            return
        yield chunk


# Example (commented):
# pipeline = batched(
#     mapper(
#         only_if(clean_rows(stream_csv(Path('data.csv'))), lambda r: r.get('qty','0').isdigit()),
#         lambda r: {**r, 'qty': int(r['qty'])}
#     ),
#     size=500
# )
# for chunk in pipeline:
#     pass  # bulk_insert(chunk)

<!-- BEGIN BEGINNER SOLUTION REVIEW -->
## Concept review before comparing answers

These worked answers demonstrate **lazy, composable data-pipeline stages with explicit resource and error policy**. Predict each named
result before comparing your attempt with its matching assertions.

A pipeline is a sequence of stages that turn input into output. Give each
stage one responsibility, read, parse, validate, normalize, filter,
batch, or write, and define the shape it accepts and yields. Small stages
are easier to test and recombine than one function mixing files, rules,
logging, and output.

Generator stages keep memory bounded by yielding one item at a time.
Calling a generator function does not run its body; iteration starts
work and advances state. The stage that opens a resource should own its
`with` block for the entire iteration. Decide whether malformed records
stop the stream, are skipped with evidence, or go to quarantine.

### Vocabulary used in the worked answers

- **pipeline:** an ordered composition of input/output stages.
- **stage:** one transformation with a stated input and output contract.
- **streaming:** processing incrementally rather than loading everything.
- **backpressure:** a consumer limiting how quickly upstream work advances.
- **batch:** a bounded group processed or written together.
- **quarantine:** separate retention of invalid records and failure reasons.

### How to compare an answer

For this lesson's **lazy, composable data-pipeline stages with explicit resource and error policy** model, follow the exact values from each learner contract through its function or expression to the assertion that proves the expected behavior; then change one boundary input and make that assertion fail once before accepting the answer.
<!-- END BEGINNER SOLUTION REVIEW -->

## Exercises 1-2: Worked answers

### Exercise 1: worked answer

**Learner contract:** Build `stream_clean_csv(path)` that opens a UTF-8 CSV inside the generator, yields one normalized row dictionary at a time, and follows a written malformed-row policy. **Constraints:** use `csv.DictReader`, keep the file open during iteration, and do not return a list. **Verify:** partially consume two rows, consume the rest, test an empty file, and confirm the handle closes after completion/failure.

**Reasoning:** A generator function's body runs only while someone iterates, so putting the `with` block inside it keeps the file open exactly while rows are being read and closes it when the loop finishes or the consumer calls `close()`. `csv.DictReader` gives each row as a dictionary keyed by the header, and `newline=""` lets the csv module handle line endings inside quoted fields correctly. The checks consume two rows, then the rest, and inspect `gi_frame`, which becomes `None` once the generator has finished or been closed: that is the evidence that the `with` block exited. The malformed-row policy, skip rows without a name or status, is written in the docstring, so behaviour for bad input is part of the contract.


In [ ]:
import csv
from collections.abc import Iterator
from pathlib import Path
import tempfile


def stream_clean_csv(path: Path) -> Iterator[dict[str, str]]:
    """Yield normalized valid rows; skip rows missing name or status."""

    with path.open(encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            if not row.get("name") or not row.get("status"):
                continue
            yield {
                "name": row["name"].strip(),
                "status": row["status"].strip().lower(),
            }


with tempfile.TemporaryDirectory() as temporary:
    folder = Path(temporary)
    source = folder / "rows.csv"
    source.write_text(
        "name,status\nAda,ACTIVE\nLin,inactive\nGrace,active\n",
        encoding="utf-8",
    )
    stream = stream_clean_csv(source)
    first = next(stream)
    second = next(stream)
    rest = list(stream)
    assert first == {"name": "Ada", "status": "active"}
    assert second == {"name": "Lin", "status": "inactive"}
    assert rest == [{"name": "Grace", "status": "active"}]
    assert stream.gi_frame is None

    partial = stream_clean_csv(source)
    assert next(partial)["name"] == "Ada"
    partial.close()
    assert partial.gi_frame is None

    empty = folder / "empty.csv"
    empty.write_text("name,status\n", encoding="utf-8")
    assert list(stream_clean_csv(empty)) == []

Because the `with` block surrounds `yield`, the file remains open while
iteration is active and closes on exhaustion or generator close.

### Exercise 2: worked answer

**Learner contract:** Compose separate generator stages to filter, map, and batch clean rows. **Contract:** every stage documents input/output shape; batch size must be positive; the final partial batch is yielded. **Verify:** on a five-row fixture with size two, assert exact batches, source order, and input = accepted + quarantined + deliberately filtered counts.

**Reasoning:** Each stage has one job with a documented input and output: `active` filters and audits, `add_label` maps one row to one new row without changing its input, and `batch` groups rows into tuples of at most `size`. Because every stage pulls one row at a time from the previous one, the whole pipeline is lazy and rows keep their source order. The audit dictionary accounts for every input row, so `accepted + filtered + quarantined == 5` proves that none were lost silently, and the final partial batch, containing only Guido, is yielded rather than dropped.


In [ ]:
from collections.abc import Iterable, Iterator


def active(
    rows: Iterable[dict[str, str]],
    audit: dict[str, int],
) -> Iterator[dict[str, str]]:
    """Map row dictionaries to accepted rows; audit other outcomes."""

    for row in rows:
        if not row.get("name") or not row.get("status"):
            audit["quarantined"] += 1
        elif row["status"] != "active":
            audit["filtered"] += 1
        else:
            audit["accepted"] += 1
            yield row


def add_label(rows: Iterable[dict[str, str]]) -> Iterator[dict[str, str]]:
    """Map one accepted row to one copied row with a display label."""

    for row in rows:
        yield {**row, "label": row["name"].title()}


def batch(rows: Iterable[dict[str, str]], size: int) -> Iterator[tuple[dict[str, str], ...]]:
    """Map a row stream to ordered tuples of at most ``size`` rows."""

    if size <= 0:
        raise ValueError("size must be positive")
    current = []
    for row in rows:
        current.append(row)
        if len(current) == size:
            yield tuple(current)
            current = []
    if current:
        yield tuple(current)


fixture = [
    {"name": "ada", "status": "active"},
    {"name": "lin", "status": "inactive"},
    {"name": "grace", "status": "active"},
    {"name": "", "status": "active"},
    {"name": "guido", "status": "active"},
]
audit = {"accepted": 0, "filtered": 0, "quarantined": 0}
result = list(batch(add_label(active(fixture, audit)), 2))
assert [[row["label"] for row in rows] for rows in result] == [
    ["Ada", "Grace"],
    ["Guido"],
]
assert audit == {"accepted": 3, "filtered": 1, "quarantined": 1}
assert len(fixture) == sum(audit.values())
assert [row["name"] for rows in result for row in rows] == ["ada", "grace", "guido"]

try:
    list(batch([], 0))
except ValueError:
    pass
else:
    raise AssertionError("non-positive batch size should fail")

## Exercises 3-7: Expanded mastery answers

### Exercise 3: answer contract

**Learner contract:** **Prediction:** Predict when side effects inside a generator body occur: at function call, first iteration, or full materialization. **Progressive hint:** Calling a generator function creates an iterator; its body starts on iteration. **Verify:** Append an event inside the generator body and assert no event at construction, one at first `next`, and all remaining events only after full consumption.

**Reasoning:** Calling `numbers(5)` only creates a generator object; none of the body runs, so the event list is still empty. The first `next` runs the body up to the first `yield` and records one event, and the remaining events appear only as later values are requested. The shared implementation for Exercises 3-4 asserts each of those three moments, which shows that side effects in a generator happen at iteration time, not at call time.

### Exercise 4: answer contract

**Learner contract:** **Tracing:** Consume two items from a five-item generator, then pass it onward. Trace which values the next stage can still see. **Progressive hint:** Consumption is stateful and does not rewind automatically. **Verify:** Assert the first consumer sees values 0 and 1 and the downstream stage sees exactly 2, 3, and 4, never a restarted sequence.

**Reasoning:** A generator remembers its position and never rewinds: after the first consumer takes 0 and 1, any later stage that receives the same object continues from 2 and sees exactly 2, 3, and 4. Once exhausted, it yields nothing more, as the final `list(stream) == []` check shows. When two stages each need the full sequence, create a fresh generator for each, or keep the values in a list if they fit in memory.

### Exercise 5: answer contract

**Learner contract:** **Implementation:** Implement `batch_rows(rows, size)` yielding tuples and preserving the final partial batch. **Progressive hint:** Validate positive size and reset the accumulator after each yield. **Verify:** Assert five rows at size two produce `[(0, 1), (2, 3), (4,)]`, empty input produces none, and nonpositive size raises.

**Reasoning:** The accumulator collects rows until it holds `size` items, yields them as a tuple, and is then cleared, so each batch starts empty; rows left over when the input ends form the final partial batch, `(4,)`. Without the size check, a size of 0 would never equal the batch length, so every row would end up in one oversized final batch. Because `batch_rows` is a generator, the check runs on the first `next`, which is why the test wraps the call in `list(...)`; the shared implementation for Exercises 5-7 asserts the three batches, the empty result for empty input, and the `ValueError` for size 0.

### Exercise 6: answer contract

**Learner contract:** **Debugging:** Repair a function that returns a generator expression over a file after the surrounding `with` block has already closed the file. **Progressive hint:** Own the `with` block inside the generator that performs iteration. **Verify:** Partially consume the repaired file generator and then finish it; assert rows remain readable until exhaustion and the handle closes afterward.

**Reasoning:** In the broken version, the `with` block closes the file as soon as the function returns the generator expression, but the expression reads from the file only later, so the first `next` raises `ValueError: I/O operation on closed file`. Moving the `with` block inside a generator function ties the file's lifetime to the iteration: it stays open while values are requested and closes when the generator finishes. The shared implementation for Exercises 5-7 reproduces the error, then consumes the repaired generator partially and fully and checks `gi_frame` afterwards.

### Exercise 7: answer contract

**Learner contract:** **Edge case and explanation:** Add a quarantine side channel for malformed rows without making the clean stream eagerly load the entire file. **Progressive hint:** Pass a callback/list for bounded error records or yield tagged results. **Verify:** Feed valid/invalid/valid rows and assert the clean stream remains lazy, output order is preserved, and accepted plus quarantined counts equal input.

**Reasoning:** Appending malformed-line evidence to a caller-supplied list keeps the clean stream lazy: the generator still yields one line at a time, and the side channel grows only when a bad line is actually read. Capping it at 100 entries keeps memory bounded even when a file is mostly bad. The shared implementation for Exercises 5-7 shows that the blank second line is not recorded until iteration reaches it, that valid lines keep their order, and that accepted plus rejected lines equal the input line count.

## Expanded mastery lab solutions

Build one-responsibility lazy stages with explicit resource ownership, error policy, and final-partial-batch behavior.

### Shared implementation for Exercises 3-4: Laziness is stateful

A generator body starts on the first `next`, not at function call. After two
items are consumed, downstream stages can see only the remaining three unless a
fresh iterator is constructed.


In [ ]:
from collections.abc import Iterator

events: list[str] = []


def numbers(limit: int) -> Iterator[int]:
    for value in range(limit):
        events.append(f"produce {value}")
        yield value


stream = numbers(5)
assert events == []  # calling the function only creates the generator
assert next(stream) == 0 and events == ["produce 0"]
assert next(stream) == 1
downstream = list(stream)  # the next stage continues from 2
assert downstream == [2, 3, 4]
assert len(events) == 5
assert list(stream) == []  # an exhausted generator does not restart

### Shared implementation for Exercises 5-7: Batching and correct resource lifetime


In [ ]:
from __future__ import annotations

from collections.abc import Iterable, Iterator
from pathlib import Path
from typing import TypeVar

T = TypeVar("T")


def batch_rows(rows: Iterable[T], size: int) -> Iterator[tuple[T, ...]]:
    """Yield tuples no larger than size, including a final partial tuple."""

    if size <= 0:
        raise ValueError("size must be positive")
    batch: list[T] = []
    for row in rows:
        batch.append(row)
        if len(batch) == size:
            yield tuple(batch)
            batch.clear()
    if batch:
        yield tuple(batch)


def nonblank_lines(path: Path, rejected: list[tuple[int, str]]) -> Iterator[str]:
    """Yield normalized lines while recording bounded malformed-line evidence."""

    # The file remains open for exactly as long as iteration needs it.
    with path.open(encoding="utf-8") as handle:
        for line_number, raw in enumerate(handle, start=1):
            normalized = raw.strip()
            if not normalized:
                if len(rejected) < 100:
                    rejected.append((line_number, "blank line"))
                continue
            yield normalized


assert list(batch_rows(range(5), 2)) == [(0, 1), (2, 3), (4,)]
assert list(batch_rows([], 2)) == []
try:
    list(batch_rows(range(3), 0))
except ValueError:
    pass
else:
    raise AssertionError("size 0 must be rejected")

import tempfile


def broken_lines(path: Path) -> Iterator[str]:
    """Bug: the with block closes the file before anyone iterates."""
    with path.open(encoding="utf-8") as handle:
        return (line.strip() for line in handle)


with tempfile.TemporaryDirectory() as temporary:
    source = Path(temporary) / "lines.txt"
    source.write_text("alpha\n\nbeta\ngamma\n", encoding="utf-8")
    try:
        next(broken_lines(source))
    except ValueError as error:
        print("broken:", error)
    else:
        raise AssertionError("the closed file should not be readable")

    rejected: list[tuple[int, str]] = []
    lines = nonblank_lines(source, rejected)
    accepted = [next(lines)]
    assert accepted == ["alpha"] and rejected == []  # line 2 not read yet
    accepted.extend(lines)
    assert accepted == ["alpha", "beta", "gamma"]  # source order kept
    assert rejected == [(2, "blank line")]
    assert lines.gi_frame is None  # finished, so the with block closed the file
    assert len(accepted) + len(rejected) == len(source.read_text(encoding="utf-8").splitlines())

The bounded quarantine prevents unbounded memory growth while leaving the
successful stream lazy. A production pipeline might yield a tagged result
object to route successes and errors to separate sinks.
